#### READ THE ORDERS PAYMENT DATA FROM VOLUME

In [0]:
BASE_DIR = '/Volumes/orders_sales/landing/operational/OrdersPayments'
CHECKPOINT_DIR = '/Volumes/orders_sales/landing/checkpoint_dir'
CHECKPOINT_LOCATION = f'{CHECKPOINT_DIR}/order_payments'
CATALOG = 'ORDERS_SALES'
SCHEMA = 'BRONZE'
TARGET_TABLE = 'ORDERS_PAYMENTS'
print(f'Checkpoint Location: {CHECKPOINT_LOCATION}')
print(f'Base Location: {BASE_DIR}')

In [0]:
orders_payments_schema = """order_id STRING, 
                 payment_sequential INT, 
                 payment_type STRING, 
                 payment_installments STRING, 
                 payment_value DOUBLE"""

#### LOAD THE DATA INTO BRONZE LAYER
- `ORDERS_SALES.BRONZE.ORDERS_PAYMENTS`

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

order_payments_df = (
            spark.readStream.format('cloudFiles')
                 .option('cloudFiles.format', 'csv')
                 .option('cloudFiles.schemaEvolutionMode', 'rescue')
                 .option('header', 'true')
                 .schema(orders_payments_schema)
                 .load(f'{BASE_DIR}')
        )
order_payments_trans_df = order_payments_df.select(
    "order_id",
    "payment_sequential",
    "payment_type",
    "payment_installments",
    "payment_value",
    "_metadata.file_name",
    "_metadata.file_path",).withColumn('load_timestamp', current_timestamp())

# order_payments_trans_df.printSchema()
# order_payments_trans_df.limit(1).display()

qry = (
    order_payments_trans_df.writeStream.format('delta')
    .outputMode('append')
    .option('checkpointLocation', CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(f'{CATALOG}.{SCHEMA}.{TARGET_TABLE}')
)
qry.awaitTermination()